In [2]:
from transformers import BertTokenizer, BertModel, AutoTokenizer
from datasets import load_dataset
import torch
import gensim.downloader
import pandas as pd
import spacy
from collections import defaultdict
import random
import json
import nltk
from nltk.corpus import wordnet as wn
from collections import Counter
from itertools import combinations

In [48]:
def load_data(path):
    dataset = load_dataset("wikitext", "wikitext-103-v1", split="train")

    glove = gensim.downloader.load("glove-wiki-gigaword-50")
    glove_voc = set(glove.key_to_index.keys())

    df = pd.read_csv(path, 
                sep="\t", 
                names=["lemma", "forms", "fam", "transf"])
    return dataset, glove_voc, df

In [49]:
def clean_dataset(dataset):
    content = []
    for line in dataset["text"]:
        line = line.strip().lower()
        if line and not line.startswith("=") and len(line.split()) > 4:
            content.append(line)
    return content

In [50]:
def get_target_words(df, glove_voc):
    target = {
            word.strip().lower()
            for form_list in df["forms"].str.split(",")
            for word in form_list
            if word.strip().lower() in glove_voc
    }
    return target

In [51]:
def get_matched_sent(content, target, json_path=None):
    if json_path:
        return pd.read_json(json_path)

    nlp = spacy.blank("en")
    nlp.add_pipe("sentencizer")
    
    matched = defaultdict(set)

    for doc in nlp.pipe(content, batch_size=2048, n_process=2):
        for sent in doc.sents:
            if len(sent) > 40:
                continue

            sent_text = sent.text
            tokens = {token.text for token in sent}

            for word in target.intersection(tokens):
                matched[word].add(sent_text)

    def tag_word(sents):
        n = len(sents)
        if n >= 50:
            return "sampled"
        elif n >= 10:
            return "low_freq"
        return "excluded"

    def sample_sents(sents):
        sents = list(sents)
        n = len(sents)
        if n >= 50:
            return random.sample(sents, 50)
        return sents

    random.seed(42)
    sent_df = pd.DataFrame(matched.items(),
                            columns=["word", "sents"])

    sent_df["tag"] = sent_df["sents"].apply(tag_word)
    sent_df["sents"] = sent_df["sents"].apply(sample_sents)

    sent_df.to_json("sent_df.json")
    print("sent_df json created")
    return sent_df

In [52]:
dataset, glove_voc, df = load_data("/home/onyxia/work/morph_families.tsv")

print("content..")
content = clean_dataset(dataset)
del dataset

print("target..")
target = get_target_words(df, glove_voc)
del df

print("matched..")
sent_df = get_matched_sent(content, target, json_path="/home/onyxia/work/sent_df.json")

content..
target..
matched..


In [53]:
sent_df.sample()

,word,sents,tag
28,written,"["" last day in florida "" was written by robert...",sampled


In [ ]:
"""def get_synonyms():
    print("get synonyms..")
    synonyms = set()
    for synset in wn.all_synsets():
        lemmas = [l.name().lower().replace("_", " ") for l in synset.lemmas()]
        for l1, l2 in combinations(lemmas, 2):
            res = tuple(sorted((l1, l2)))
            synonyms.add(res)

    with open("/home/onyxia/work/synonyms.json", "w") as f:
        json.dump(list(synonyms), f)

    return list(synonyms)


def get_antonyms():
    print("get antonyms..")
    antonyms = set()
    for synset in wn.all_synsets():
        for lemma in synset.lemmas():
            word = lemma.name().lower().replace("_", " ")
            for ant in lemma.antonyms():
                word_ant = ant.name().lower().replace("_", " ")
                res = tuple(sorted((word, word_ant)))
                antonyms.add(res)

    with open("/home/onyxia/work/antonyms.json", "w") as f:
        json.dump(list(antonyms), f)

    return list(antonyms)


def filter_word(pair_set, content, glove_voc, word_counts, filename):
    print("filter words..")
    random.seed(42)
    filtered = []
    
    for l1, l2 in pair_set:
        if l1 in glove_voc and l2 in glove_voc:
            if word_counts[l1] >= 10 and word_counts[l2] >= 10:
                filtered.append((l1, l2))

    if len(filtered) > 200:
        filtered = random.sample(filtered, 200)
    else:
        print(f"{len(filtered)} pairs found")
    
    with open(f"/home/onyxia/work/filtered_{filename}.json", "w") as f:
        json.dump(filtered, f)
        
    return list(filtered)


nltk.download("wordnet")
word_counts = Counter()
for sentence in content:
    word_counts.update(sentence.split())

synonyms = filter_word(get_synonyms(), content, glove_voc, word_counts, filename="synonyms")
antonyms = filter_word(get_antonyms(), content, glove_voc, word_counts, filename="antonyms")"""

[nltk_data] Downloading package wordnet to /home/onyxia/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


get synonyms..
filter words..
get antonyms..
filter words..


In [ ]:
with open("/home/onyxia/work/filtered_synonyms.json", "r") as f:
    synonyms = json.load(f)

with open("/home/onyxia/work/filtered_antonyms.json", "r") as f:
    antonyms = json.load(f)

In [ ]:
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

CUDA: True
GPU : NVIDIA A2


In [3]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
encoding = tokenizer(example)
print(type(encoding))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [8]:
tokenizer.is_fast
encoding.is_fast
encoding.tokens()
encoding.word_ids()
start, end = encoding.word_to_chars(3)
example[start:end]

'Sylvain'